# CSV → JSON & XML Converter

This notebook demonstrates how to read a CSV file, convert it to JSON and XML, and write the results to disk.  
All steps are fully documented so you can adapt the code to your own data.

---

## 1️⃣ Load the CSV

In [ ]:
import os
import pandas as pd
import json
import xml.etree.ElementTree as ET

repo_root = os.path.dirname(os.getcwd())
DATA_PATH_ES = os.path.join(repo_root,'data/raw/', 'es')

# Load your dataset
csv_file_path = os.path.join(DATA_PATH_ES, 'data.csv')
df = pd.read_csv(csv_file_path)
df.head()

## 2️⃣ Convert to JSON

In [ ]:
# Convert the DataFrame to a list of dictionaries
records = df.to_dict(orient="records")

# Pretty‑print the JSON to a file
json_path = os.path.join(DATA_PATH_ES, 'data.json')
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print(f"✅ JSON written to {json_path}")

## 3️⃣ Convert to XML

In [ ]:
def dict_to_xml(tag, d):
    """Recursively convert a dict to an ElementTree Element."""
    elem = ET.Element(tag)
    for key, val in d.items():
        if isinstance(val, dict):
            child = dict_to_xml(key, val)
            elem.append(child)
        elif isinstance(val, list):
            for item in val:
                child = dict_to_xml(key, item if isinstance(item, dict) else {"value": item})
                elem.append(child)
        else:
            child = ET.Element(key)
            child.text = str(val)
            elem.append(child)
    return elem

# Build the XML tree
root = dict_to_xml("dataset", {"record": records})
xml_path = os.path.join(DATA_PATH_ES, "data.xml")

# Pretty‑print the tree (Python 3.9+)
try:
    ET.indent(root, space="  ", level=0)          # two‑space indent
except AttributeError:
    # For older Python versions – use minidom
    import xml.dom.minidom
    rough_string = ET.tostring(root, encoding="utf-8")
    reparsed = xml.dom.minidom.parseString(rough_string)
    pretty_xml = reparsed.toprettyxml(indent="  ")
    with open(xml_path, "w", encoding="utf-8") as f:
        f.write(pretty_xml)
    print(f"✅ XML written to {xml_path} (pretty‑printed)")
    exit()

# Write the indented XML
ET.ElementTree(root).write(xml_path, encoding="utf-8", xml_declaration=True)

print(f"✅ XML written to {xml_path} (pretty‑printed)")